# SOFR prediction + email preview (Jupyter)

1. `git pull origin main`
2. Corre setup → opciones (`SEND_EMAIL = False`) → predicción → **preview HTML**
3. Solo cuando el preview esté bien, pon `SEND_EMAIL = True` y rellena SMTP

El preview muestra **exactamente** el HTML del email (tabla + gráficos) sin SMTP.

In [ ]:
import importlib
import logging
import os
import sys
from pathlib import Path

from IPython.display import HTML, Image, display
import pandas as pd

logging.disable(logging.WARNING)
logging.getLogger().setLevel(logging.ERROR)
for _n in list(logging.root.manager.loggerDict):
    lg = logging.getLogger(_n)
    if any(k in str(_n).lower() for k in ("bbg", "blp", "dz", "datazone", "activity", "ms")):
        lg.disabled = True
        lg.handlers = []

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 20)

_DZ = Path("/v/region/na/appl/datazone/notebooks2/data/users/arraipau/fwd_screener_v1")
REPO = Path.cwd() if (Path.cwd() / "run_prediction.py").exists() else _DZ
if not (REPO / "run_prediction.py").exists():
    raise FileNotFoundError("No encuentro run_prediction.py en %s" % REPO)

os.chdir(REPO)
sys.path = [str(REPO)] + [p for p in sys.path if str(Path(p).resolve()) != str(REPO.resolve())]
print("REPO:", REPO.resolve())
print("python:", sys.executable)

In [ ]:
import run_dz

importlib.reload(run_dz)
run_dz.bootstrap_for_repo(str(REPO))

In [ ]:
# ── Opciones ────────────────────────────────────────────────────────────────
REFRESH = False
PULL_HISTORY = False
SEND_EMAIL = False       # False = solo preview en notebook (sin SMTP)
MAKE_GRAPHS = True

import curve_config

importlib.reload(curve_config)
print("SEND_EMAIL:", SEND_EMAIL)
print("GRAPH_LOOKBACK_DAYS:", curve_config.GRAPH_LOOKBACK_DAYS)

In [ ]:
# Predicción (+ graphs). No envía si SEND_EMAIL=False.
import data
import run_prediction

importlib.reload(data)
importlib.reload(run_prediction)

table = run_prediction.run_once(
    refresh=REFRESH,
    force_history=PULL_HISTORY,
    send_email=SEND_EMAIL,
    make_graphs=MAKE_GRAPHS,
)

from models import PREDICTION_COLS

cols = [c for c in PREDICTION_COLS if c in table.columns]
display(table[cols])

In [ ]:
# ── Preview del email (HTML = lo que llegaría a Gmail) ───────────────────────
html = run_prediction.preview_email_html()
print("asof:", run_prediction.LAST_ASOF, "graphs:", len(run_prediction.LAST_GRAPH_PATHS))
display(HTML(html))

# opcional: guardar HTML local para abrirlo en el navegador
out = REPO / "email_preview.html"
out.write_text(html, encoding="utf-8")
print("wrote", out.resolve())

## Enviar de verdad (opcional)

Cuando el preview esté bien: pon `SEND_EMAIL = True` arriba, rellena SMTP en la celda siguiente y vuelve a correr predicción.

In [ ]:
# Solo si vas a enviar de verdad
if SEND_EMAIL:
    curve_config.EMAIL_ENABLED = True
    curve_config.SMTP_HOST = "smtp.gmail.com"
    curve_config.SMTP_PORT = 587
    curve_config.SMTP_USE_TLS = True
    curve_config.SMTP_USER = ""
    curve_config.SMTP_PASSWORD = ""
    curve_config.REPORT_FROM = ""
    curve_config.REPORT_TO = []

    import models

    importlib.reload(models)
    for k in (
        "EMAIL_ENABLED", "SMTP_HOST", "SMTP_PORT", "SMTP_USER", "SMTP_PASSWORD",
        "REPORT_FROM", "SMTP_USE_TLS",
    ):
        setattr(models, k, getattr(curve_config, k))
    models.REPORT_TO = list(curve_config.REPORT_TO)

    importlib.reload(run_prediction)
    run_prediction.run_once(
        refresh=REFRESH, force_history=PULL_HISTORY, send_email=True, make_graphs=MAKE_GRAPHS,
    )
else:
    print("SEND_EMAIL=False → no SMTP. Cambia a True y rellena credenciales para enviar.")